# Notebook 09 - Deadline Final Ensemble (Phase 2)

Objectif: produire des soumissions **très compétitives** sans relancer des entraînements lourds.

Méthodes intégrées:
- **Weighted RRF multi-runs** (fusion des meilleures soumissions existantes),
- **Routing par confiance de classification** (pondérations différentes selon la difficulté de la requête),
- **Boost de cohérence catégorie** (repondération des documents ayant la même catégorie prédite que la query).


## 1) Imports et chargement


In [1]:
import os
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

DATA_DIR = "../data"
CHAT_OUTPUTS_DIR = "../outputs"
VAL_OUTPUTS_DIR = "../../val/outputs"

# Docs + queries
with open(os.path.join(DATA_DIR, "queries_train.json"), "r", encoding="utf-8") as f:
    queries_train = json.load(f)
with open(os.path.join(DATA_DIR, "queries_test.json"), "r", encoding="utf-8") as f:
    queries_test = json.load(f)

df_docs = pd.read_pickle(os.path.join(DATA_DIR, "df_docs_preprocessed.pkl"))
doc_cat_by_id = dict(zip(df_docs["id"], df_docs["category"]))

print(f"Train queries: {len(queries_train)}")
print(f"Test queries : {len(queries_test)}")
print(f"Docs         : {len(df_docs):,}")


Train queries: 327
Test queries : 141
Docs         : 216,041


## 2) Classifieur de catégorie (pour routing + catégorie finale)


In [2]:
def build_query_text_lex(q):
    return " ".join([
        q.get("text", "") or "",
        " ".join(q.get("tags", []) or []),
        q.get("title", "") or "",
    ]).strip()

# sample docs (rapide)
sampled_docs = []
for cat, group in df_docs.groupby("category"):
    sampled_docs.append(group.sample(n=min(len(group), 9000), random_state=42))
df_docs_sampled = pd.concat(sampled_docs, ignore_index=True)

X_docs = df_docs_sampled["content_clean"].astype(str).tolist()
y_docs = df_docs_sampled["category"].astype(str).tolist()
X_q = [build_query_text_lex(q) for q in queries_train]
y_q = [q["category"] for q in queries_train]

classifier = make_pipeline(
    TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=280000,
        sublinear_tf=True,
    ),
    LogisticRegression(max_iter=1000, C=4.0, solver="saga", n_jobs=-1),
)

classifier.fit(X_docs + X_q, y_docs + y_q)

X_test_lex = [build_query_text_lex(q) for q in queries_test]
test_pred_cat = classifier.predict(X_test_lex)
test_pred_proba = classifier.predict_proba(X_test_lex)

print("Classifier ready")
print("Classes:", list(classifier.classes_))


Classifier ready
Classes: [np.str_('android'), np.str_('gaming'), np.str_('programmers'), np.str_('tex'), np.str_('unix')]


## 3) Chargement des meilleures runs existantes


In [3]:
RUN_FILES = {
    "v8": os.path.join(VAL_OUTPUTS_DIR, "92iemj_v8_hybrid_cv.csv"),
    "p2f": os.path.join(VAL_OUTPUTS_DIR, "submission_phase2_final.csv"),
    "v5": os.path.join(VAL_OUTPUTS_DIR, "92iemj_v5_crossencoder.csv"),
    "v7": os.path.join(VAL_OUTPUTS_DIR, "92iemj_v7_tuned.csv"),
    "base92": os.path.join(VAL_OUTPUTS_DIR, "92iemj.csv"),
    "ourv1": os.path.join(CHAT_OUTPUTS_DIR, "phase2_submission_v1.csv"),
}


def load_sub(path):
    df = pd.read_csv(path)
    df["relevant_doc_ids"] = df["relevant_doc_ids"].apply(json.loads)
    return df


runs = {}
for name, path in RUN_FILES.items():
    if os.path.exists(path):
        runs[name] = load_sub(path)
        print(f"[OK] {name:6s} -> {os.path.basename(path)}")
    else:
        print(f"[WARN] missing: {path}")

base_name = next(iter(runs))
base_qids = runs[base_name]["query_id"].tolist()
for name, df_run in runs.items():
    if df_run["query_id"].tolist() != base_qids:
        raise ValueError(f"Query order mismatch: {name}")

run_q2docs = {name: dict(zip(df["query_id"], df["relevant_doc_ids"])) for name, df in runs.items()}
cat_fallback = dict(zip(runs["p2f"]["query_id"], runs["p2f"]["category"])) if "p2f" in runs else {}


[OK] v8     -> 92iemj_v8_hybrid_cv.csv
[OK] p2f    -> submission_phase2_final.csv
[OK] v5     -> 92iemj_v5_crossencoder.csv
[OK] v7     -> 92iemj_v7_tuned.csv
[OK] base92 -> 92iemj.csv
[OK] ourv1  -> phase2_submission_v1.csv


## 4) Weighted RRF + routing + category boost


In [4]:
def weighted_rrf(rankings, weights, k_rrf=20):
    scores = defaultdict(float)
    for docs, w in zip(rankings, weights):
        for rank, doc_id in enumerate(docs, start=1):
            scores[doc_id] += float(w) / (k_rrf + rank)
    return scores


def generate_deadline_submission(file_name, mode="router1"):
    rows = []
    for i, q in enumerate(queries_test):
        qid = q["id"]
        conf = float(np.max(test_pred_proba[i]))
        pred_cat = test_pred_cat[i]

        if mode == "router1":
            if conf >= 0.92:
                chosen = ["p2f", "v5", "v8", "v7"]
                weights = [2.0, 1.5, 1.0, 0.6]
                k_rrf = 15
            elif conf >= 0.75:
                chosen = ["v8", "p2f", "v5", "ourv1"]
                weights = [2.0, 1.2, 1.0, 0.5]
                k_rrf = 20
            else:
                chosen = ["v8", "v7", "base92", "p2f"]
                weights = [2.0, 1.0, 0.8, 0.8]
                k_rrf = 25
            beta = 0.22

        elif mode == "router2":
            if conf >= 0.90:
                chosen = ["v5", "p2f", "v8"]
                weights = [2.0, 1.6, 1.0]
                k_rrf = 12
            elif conf >= 0.70:
                chosen = ["v8", "p2f", "v7", "ourv1"]
                weights = [2.2, 1.0, 0.8, 0.5]
                k_rrf = 18
            else:
                chosen = ["v8", "base92", "v7"]
                weights = [2.5, 1.0, 1.0]
                k_rrf = 28
            beta = 0.18

        elif mode == "ultra":
            chosen = ["v8", "p2f", "v5", "v7", "ourv1", "base92"]
            weights = [2.3, 1.3, 1.1, 0.8, 0.6, 0.4]
            k_rrf = 16
            beta = 0.20

        else:
            raise ValueError(mode)

        chosen = [c for c in chosen if c in run_q2docs]
        rankings = [run_q2docs[c][qid] for c in chosen]
        weights = weights[:len(chosen)]

        score_dict = weighted_rrf(rankings, weights, k_rrf=k_rrf)

        # category-aware boost on fused candidates
        for doc_id in list(score_dict.keys()):
            if doc_cat_by_id.get(doc_id) == pred_cat:
                score_dict[doc_id] *= (1.0 + beta * conf)

        fused = sorted(score_dict.keys(), key=lambda d: score_dict[d], reverse=True)[:100]

        rows.append({
            "query_id": qid,
            "relevant_doc_ids": json.dumps(fused),
            "category": pred_cat if pred_cat else cat_fallback.get(qid, "tex"),
        })

    out = os.path.join(CHAT_OUTPUTS_DIR, file_name)
    df_out = pd.DataFrame(rows)

    # format sanity checks (not benchmark)
    assert len(df_out) == len(queries_test)
    assert (df_out["relevant_doc_ids"].apply(lambda x: len(json.loads(x))) == 100).all()

    df_out.to_csv(out, index=False)
    print(f"[OK] saved {out}")


In [5]:
generate_deadline_submission("phase2_deadline_router_v1.csv", mode="router1")
generate_deadline_submission("phase2_deadline_router_v2.csv", mode="router2")
generate_deadline_submission("phase2_deadline_rrf_ultra.csv", mode="ultra")


[OK] saved ../outputs/phase2_deadline_router_v1.csv
[OK] saved ../outputs/phase2_deadline_router_v2.csv
[OK] saved ../outputs/phase2_deadline_rrf_ultra.csv


## Soumissions recommandées (ordre)

1. `phase2_deadline_router_v1.csv`
2. `phase2_deadline_rrf_ultra.csv`
3. `phase2_deadline_router_v2.csv`
